In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Potential Energy

Last time we found integrators that hold onto the energy for as long as we care to run them.  But there was some fine print: symplectic Euler and velocity Verlet shine when the force depends only on *position*, like a spring, gravity, or a pendulum.  Add drag and the magic is gone.

So what do all those position-only forces have in common?  And where did we get the energy formulas we've been checking, anyway?  For the oscillator and the pendulum we just wrote them down.

Both questions have the same answer: **potential energy**.  Once we have it, we'll be able to describe a whole physical system by drawing one curve, and we'll be able to tell what the system is going to do just by looking at it, before we integrate anything!

## Our Tools So Far

Here's what we need from last time, copied over without changes.  (Skim past them if you remember them!)  We only need one integrator today, and we know which one we trust.

In [ ]:
class FirstOrderODE:
    '''
    A first-order ODE specifies the rate of change of a state y(t)
    in terms of
     - the state itself, y
     - the time, t.

    The generic form for an ODE is

    dy
    --  = f(t, y)
    dt

    '''

    def f(self, t, y):
        raise NotImplementedError('The generic ODE has an unspecified slope function f.')

    def exact_solution(self, t, *args):
        raise NotImplementedError('The exact solution is not known (or at least not provided).')

In [ ]:
class SecondOrderODE(FirstOrderODE):
    '''
    A second-order ODE specifies the acceleration

    d^2x
    ----  = a(t, x, v)
    dt^2

    where v = dx/dt is the velocity.  Treating the state as y = (x, v) turns it into
    a first-order ODE,

    d  ( x )   (    v     )
    -- (   ) = (          )
    dt ( v )   ( a(t,x,v) )

    so a subclass only needs to provide the acceleration.
    '''

    def acceleration(self, t, x, v):
        raise NotImplementedError('The generic second-order ODE has an unspecified acceleration.')

    def f(self, t, y):
        # Unpack the state into its two components,
        x, v = y
        # and pack their rates of change back up the same way.
        return np.array([v, self.acceleration(t, x, v)])

In [ ]:
class FirstOrderODEIntegrator:
    '''
    An integrator is an algorithm for producing numerical solutions to ODEs.
    '''

    def __init__(self, ODE):

        self.ODE = ODE

    def starting_from(self, t_0, y_0):
        raise NotImplementedError('No integration algorithm has been specified.')

In [ ]:
class VelocityVerlet(FirstOrderODEIntegrator):
    '''
    Velocity Verlet splits each step into a half kick, a drift, and another half kick,

    v_{n+1/2} = v_n       + 1/2 a(t_n, x_n) Δt
    x_{n+1}   = x_n       + v_{n+1/2} Δt
    v_{n+1}   = v_{n+1/2} + 1/2 a(t_{n+1}, x_{n+1}) Δt

    It needs the ODE's acceleration, so it only works on second-order ODEs.
    '''

    def __init__(self, ODE, Delta_t, steps):

        self.Delta_t = Delta_t
        self.steps = steps

        super().__init__(ODE)

    def starting_from(self, t_0, y_0):

        # Populate all the t values
        t = t_0 + self.Delta_t * np.arange(self.steps+1)
        # and allocate an array with room for a whole state at each time.
        y = np.zeros((self.steps+1,) + np.shape(y_0))

        # The user picked a starting state
        y[0] = y_0

        for step in range(self.steps):
            x, v = y[step]
            # (acceleration also wants a velocity; we hand it the newest one we have.
            #  For forces that depend only on position it doesn't matter.)
            v_half = v      + 0.5 * self.ODE.acceleration(t[step],   x,     v)      * self.Delta_t   # half kick
            x_new  = x      +       v_half                                           * self.Delta_t   # drift
            v_new  = v_half + 0.5 * self.ODE.acceleration(t[step+1], x_new, v_half) * self.Delta_t   # half kick
            y[step+1] = np.array([x_new, v_new])

        return t, y

# Two Energies, One Pattern

Here are the two `energy` methods we've written so far.

```python
# HarmonicOscillator
return 0.5 * v**2 + 0.5 * self.omega**2 * x**2

# Pendulum
return 0.5 * v**2 + self.omega**2 * (1 - np.cos(x))
```

See the pattern?  Each one is $\frac{1}{2}v^2$ plus *something that only depends on $x$*.

These are really energies *per unit mass*: we divided out the mass so there'd be less to type.  For a little while, let's put the mass back.  Then the first piece is the kinetic energy $\frac{1}{2}mv^2$, the energy of motion.  Let's give the second piece a name, $V(x)$, and call it the **potential energy**:
\begin{align}
    E &= \frac{1}{2} m v^2 + V(x).
\end{align}
For our two systems
\begin{align}
    V_\text{oscillator}(x) &= \frac{1}{2} m \omega^2 x^2 = \frac{1}{2} k x^2
    &
    V_\text{pendulum}(\theta) &= m g \ell (1 - \cos\theta).
\end{align}
The oscillator's is the familiar energy stored in a spring, since $\omega^2 = k/m$.  The pendulum's makes good sense too: the bob sits a height $\ell(1-\cos\theta)$ above the bottom of its swing, so this is just $mgh$.

(For the pendulum $x$ is the angle $\theta$ and $v$ is the angular velocity, so the kinetic energy is $\frac{1}{2}m\ell^2 v^2$.  There the moment of inertia $m\ell^2$ plays the part of the mass.)

## When Is the Energy Conserved?

Suppose somebody hands us *any* function $V(x)$ and we build $E = \frac{1}{2}mv^2 + V(x)$ out of it.  Does $E$ stay constant as the system moves?  Let's take its time derivative and see.  Both $x$ and $v$ change with time, so we need the chain rule:
\begin{align}
    \frac{dE}{dt}
    &= m v \frac{dv}{dt} + V'(x) \frac{dx}{dt}
    \\
    &= m v\, a + V'(x)\, v
    \\
    &= v \left( m a + V'(x) \right).
\end{align}
We want that to be zero no matter how fast the thing is moving.  There's only one way:
\begin{align}
    m a &= -V'(x).
\end{align}

That's Newton's second law, with the force $F = -V'(x)$.  The force is *minus the slope of the potential energy*.

Let's check it on the two systems we know.
 - Oscillator: $V = \frac{1}{2}kx^2$ has slope $V' = kx$, so $ma = -kx$.  That's Hooke's law!
 - Pendulum: $V = mg\ell(1-\cos\theta)$ has slope $V' = mg\ell\sin\theta$.  Its "mass" is $m\ell^2$, so $m\ell^2 a = -mg\ell\sin\theta$, or $a = -(g/\ell)\sin\theta = -\omega^2\sin\theta$.  That's the pendulum!

So the potential energy isn't an extra fact about a system that we have to remember alongside its force.  It *is* the force, packaged differently.  Give me $V$ and I can differentiate to get $F$.  Give me $F(x)$ and I can integrate to get $V$.

**From here on, everything is per unit mass.**  The mass just goes along for the ride, so let's go back to what we were doing all along: divide it out (by the moment of inertia $m\ell^2$ for the pendulum), and use $V$ for the potential energy *per unit mass*.  Then
\begin{align}
    E &= \frac{1}{2} v^2 + V(x)
    &
    a &= -V'(x),
\end{align}
and our two potentials are
\begin{align}
    V_\text{oscillator}(x) &= \frac{1}{2}\omega^2 x^2
    &
    V_\text{pendulum}(\theta) &= \omega^2 (1 - \cos\theta),
\end{align}
which are exactly the pieces in the `energy` methods above.

Two things to notice.

**Only differences in $V$ matter.**  If we add a constant to $V$ its slope doesn't change, so the motion doesn't change either.  We get to pick where $V=0$.  For the pendulum we picked the bottom of the swing.

**Not every force has a potential.**  We needed the force to depend only on $x$.  Drag depends on $v$, so there's no $V$ for it, and no conserved energy to go with it.  That's the real reason drag was in the fine print last time.  A force that comes from a potential is called a **conservative** force, because it conserves the energy.

# Conservative Systems as Classes

Every one of these systems computes its energy the same way: $\frac{1}{2}v^2$ plus its own potential.  So let's write that *once*, in a class that sits between `SecondOrderODE` and the specific systems.  A specific system now tells us two things, its `potential` and its `acceleration`, and it gets `energy` for free.

In [ ]:
class ConservativeSystem(SecondOrderODE):
    '''
    A conservative system has a potential energy V(x), and its acceleration is

    a = -V'(x)

    which doesn't depend on the velocity or the time.  Its energy

    E = 1/2 v^2 + V(x)

    is conserved.  A subclass provides the potential and the acceleration (minus its slope!).
    Everything is per unit mass.
    '''

    def potential(self, x):
        raise NotImplementedError('The generic conservative system has an unspecified potential.')

    def energy(self, x, v):
        return 0.5 * v**2 + self.potential(x)

Here are our two old friends again.  Their `energy` methods are gone, and each has a `potential` instead.

In [ ]:
class HarmonicOscillator(ConservativeSystem):
    '''
    A harmonic oscillator has potential V = 1/2 ω^2 x^2 and acceleration a = -ω^2 x.
    '''

    def __init__(self, omega):
        self.omega = omega

    def potential(self, x):
        return 0.5 * self.omega**2 * x**2

    def acceleration(self, t, x, v):
        return -self.omega**2 * x

    def exact_solution(self, t, x_0, v_0):
        x =  x_0 * np.cos(self.omega * t) + v_0 / self.omega * np.sin(self.omega * t)
        v = -x_0 * self.omega * np.sin(self.omega * t) + v_0 * np.cos(self.omega * t)
        return np.array([x, v])

In [ ]:
class Pendulum(ConservativeSystem):
    '''
    A simple pendulum has potential V = ω^2 (1 - cos(θ)) and angular acceleration a = -ω^2 sin(θ),
    where ω^2 = g/ℓ.  We use x for the angle θ and v for the angular velocity.
    '''

    def __init__(self, omega):
        self.omega = omega

    def potential(self, x):
        return self.omega**2 * (1 - np.cos(x))

    def acceleration(self, t, x, v):
        return -self.omega**2 * np.sin(x)

We never wrote an `energy` method for either one, but they both have it.

In [ ]:
oscillator = HarmonicOscillator(1.0)
pendulum = Pendulum(1.0)

print(oscillator.energy(1.0, 0.0))
print(pendulum.energy(np.pi, 0.0))

An oscillator released from $x=1$ has energy $\frac{1}{2}$, and a pendulum balanced upside-down at $\theta = \pi$ has energy $2\omega^2$ (it's $2\ell$ above the bottom).

# Rolling Downhill

Let's draw the pendulum's potential, and on top of it draw the acceleration at a few different angles.

In [ ]:
theta = np.linspace(-2*np.pi, 2*np.pi, 400)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(theta, pendulum.potential(theta), color='black')

# The acceleration at a few angles, drawn as arrows sitting on the curve.
# (The velocity and the time don't matter, so we hand the acceleration zeros.)
here = np.array([-5.5, -4.2, -2.2, -0.9, 0.9, 2.2, 4.2, 5.5])
ax.quiver(here, pendulum.potential(here), pendulum.acceleration(0, here, 0), np.zeros_like(here),
          color='C3', angles='xy', scale_units='xy', scale=1, width=0.004, zorder=3)

ax.set_xticks(np.pi * np.arange(-2, 3), ['-2π', '-π', '0', 'π', '2π'])
ax.set_xlabel('θ')
ax.set_ylabel('V(θ)')

Every arrow points *downhill*.  That's what the minus sign in $a = -V'$ does: where the curve slopes up to the right, the push is to the left.  And the steeper the hill, the longer the arrow.

So here's a great way to think about any potential: imagine a ball rolling on a landscape with that shape.

Where the curve is flat there's no force at all.  Those spots are the *equilibria*, and there are two kinds.
 - At the bottom of a valley ($\theta = 0$, hanging straight down) the arrows on both sides point back toward the bottom.  Nudge the pendulum and it comes back.  That's a *stable* equilibrium.
 - At the top of a hill ($\theta = \pm\pi$, balanced upside-down) the arrows on both sides point away.  Nudge it and it falls.  That's an *unstable* equilibrium.

(The valleys at $\theta = \pm 2\pi$ are the same valley as $\theta = 0$: the pendulum has gone all the way around once.)

# Trust, but Verify

Our classes ask for the potential *and* the acceleration, and it's on us to make sure they agree.  It's really easy to drop a minus sign!  Can we get the computer to check our calculus?

The slope of a curve at $x$ is very nearly the slope of the straight line through two points on the curve, one a tiny bit to the left of $x$ and one a tiny bit to the right:
\begin{align}
    V'(x) &\approx \frac{V(x+\Delta x) - V(x-\Delta x)}{2\,\Delta x}.
\end{align}
This is called a *central difference*.  If we Taylor expand the right-hand side in $\Delta x$ we find
\begin{align}
    \frac{V(x+\Delta x) - V(x-\Delta x)}{2\,\Delta x} &= V'(x) + \frac{1}{6} V'''(x)\, \Delta x^2 + \cdots
\end{align}
so the error shrinks like $\Delta x^2$.

We want $\Delta x$ small, so the curve doesn't have room to bend between the two points.  But not *too* small: then $V(x+\Delta x)$ and $V(x-\Delta x)$ agree in almost all of their digits, and when we subtract them we're left with mostly roundoff.  Something like $\Delta x = 10^{-5}$ is a good compromise.

In [ ]:
def slope(function, x, Delta_x=1e-5):
    '''
    Estimates the derivative of a function at x with a central difference.
    '''
    return (function(x + Delta_x) - function(x - Delta_x)) / (2 * Delta_x)

def acceleration_matches_potential(system, x):
    '''
    Checks whether a conservative system's acceleration is minus the slope of its potential at every x.
    '''
    return np.allclose(system.acceleration(0, x, 0), -slope(system.potential, x))

Notice that we hand `slope` the *method itself*, `system.potential` with no parentheses, so that `slope` can call it wherever it likes.

In [ ]:
x = np.linspace(-3, 3, 13)

print(pendulum.acceleration(0, x, 0))
print(-slope(pendulum.potential, x))

print(acceleration_matches_potential(oscillator, x))
print(acceleration_matches_potential(pendulum, x))

Both of our systems pass.  Does the check actually catch mistakes?  Let's make one on purpose.

In [ ]:
class SloppyPendulum(Pendulum):
    '''
    A pendulum whose author forgot a minus sign.
    '''

    def acceleration(self, t, x, v):
        return self.omega**2 * np.sin(x)

print(acceleration_matches_potential(SloppyPendulum(1.0), x))

Caught!  Run this check every time you write a new conservative system.  It takes one line, and it's a lot more pleasant than staring at a simulation that does something weird and wondering why.

# Reading a Potential

Now for the payoff.  The kinetic energy $\frac{1}{2}v^2$ can never be negative, and $E = \frac{1}{2}v^2 + V(x)$, so
\begin{align}
    V(x) &\leq E.
\end{align}
A system with energy $E$ can only ever be found at places where the potential is below $E$.  That's it.  That one little inequality tells us a lot!

Draw the energy as a horizontal line on top of the potential.
 - Wherever the line is above the curve, the system is allowed to be there, and the gap between them is the kinetic energy.  A bigger gap means faster: $v = \pm\sqrt{2(E - V(x))}$.
 - Where the line *meets* the curve, all the energy is potential and the speed is zero.  The system stops and turns around, so these are called **turning points**.
 - Where the line is below the curve, the system can never go.

For the pendulum the hilltops are at height $2\omega^2$.  If $E$ is less than that, the line hits the curve on both sides of the valley and the pendulum is trapped between two turning points, swinging back and forth.  It's **bound**.  If $E$ is more than that, the line clears the hilltops, there are no turning points at all, and the pendulum goes around and around forever.

Let's test those predictions.  We'll start five pendulums at the bottom, where $V=0$ and all the energy is kinetic, so a pendulum with energy $E$ starts with $v = \sqrt{2E}$.  Four have less energy than the hilltop (with $\omega = 1$, that's 2) and one has more.

In [ ]:
E = np.array([0.5, 1.0, 1.5, 1.9, 2.5])

theta_0 = np.zeros_like(E)
v_0 = np.sqrt(2 * E)          # at the bottom all of the energy is kinetic
y_0 = np.array([theta_0, v_0])

t, y = VelocityVerlet(pendulum, 0.01, 2000).starting_from(0, y_0)
print(y.shape)

Before we look, let's predict the turning points.  They're where $V(\theta) = E$, which for the pendulum means $\omega^2(1 - \cos\theta) = E$, or $\theta = \pm\arccos(1 - E/\omega^2)$.  That only has an answer for the four bound pendulums.

In [ ]:
bound = E < pendulum.potential(np.pi)
turning_point = np.arccos(1 - E[bound] / pendulum.omega**2)

print('predicted turning points:', turning_point)
print('largest angle reached:   ', np.max(y[:, 0, bound], axis=0))

The integrator never heard about turning points, but each bound pendulum swings out to its turning point and stops there, to four decimal places.

Here's the whole story in one picture.  On top is the potential with each pendulum's energy drawn over the angles it's allowed to visit.  Below is phase space, with the trajectories velocity Verlet found.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

axes[0].plot(theta, pendulum.potential(theta), color='black')

# The bound pendulums live between their turning points,
for i, edge in enumerate(turning_point):
    axes[0].plot([-edge, edge], [E[i], E[i]], color=f'C{i}', label=f'E = {E[i]}')
    for axis in axes:
        axis.axvline(-edge, color=f'C{i}', linestyle='dotted', alpha=0.5)
        axis.axvline(+edge, color=f'C{i}', linestyle='dotted', alpha=0.5)

# and the one with enough energy to clear the hilltops can go anywhere.
for i in np.flatnonzero(~bound):
    axes[0].axhline(E[i], color=f'C{i}', label=f'E = {E[i]}')

for i in range(len(E)):
    axes[1].plot(y[:, 0, i], y[:, 1, i], color=f'C{i}')

axes[0].set_ylabel('V(θ)')
axes[0].legend(loc='upper center', ncols=len(E))
axes[0].set_ylim(0, 3.2)

axes[1].set_xlim(-2*np.pi, 2*np.pi)
axes[1].set_xticks(np.pi * np.arange(-2, 3), ['-2π', '-π', '0', 'π', '2π'])
axes[1].set_xlabel('θ')
axes[1].set_ylabel('v')

Follow the dotted lines down from the top panel to the bottom one.  Each bound pendulum's loop in phase space reaches exactly as far as its energy line does, and crosses $v=0$ right at its turning points.  The loops are tallest at $\theta=0$, where the gap between the energy and the potential is biggest.

The small loops look like the oscillator's ellipses, because near the bottom the valley looks like a parabola.  The big ones get pinched toward the hilltops at $\pm\pi$, where the pendulum slows to a crawl and lingers before falling back.

And the pendulum with $E = 2.5$ never turns around.  It slows down as it goes over each hilltop, speeds up through each valley, and sails off the right edge of the picture.

In fact the bottom panel is just the top panel in disguise: each curve is $v = \pm\sqrt{2(E - V(\theta))}$.  We could have drawn phase space without integrating anything!  What the integrator adds is *when*: how long it takes to get around.

Since we let velocity Verlet do the work, the energies should have stayed put.  Our inherited `energy` method broadcasts over all five pendulums at once.

In [ ]:
energy = pendulum.energy(y[:, 0], y[:, 1])
print(energy.shape)
print('largest change in E / E(0):', np.max(np.abs(energy / energy[0] - 1), axis=0))

All five are good to a few parts in a hundred thousand.

That matters more than it might seem.  Look at the pendulum with $E = 1.9$: it's bound, but only barely.  An integrator that lets the energy creep upward would eventually carry it over the hilltop, and it would start spinning around and around, which is a completely different kind of motion!  (That's exactly what happened to Euler's pendulum last time.)  When a system is close to the edge between two behaviors, conserving energy is the difference between the right answer and nonsense.

# What About More Dimensions?

Everything today lived on a line: one position $x$, one velocity $v$, and a potential that's a curve.  That let us

 - find the acceleration from the potential, $a = -V'(x)$,
 - check our calculus with a central difference,
 - and read off where the system can go, where it turns around, and whether it's bound.

But a planet doesn't live on a line.  Its position has three components, and its potential energy depends on all of them.  The landscape isn't a curve anymore.  So what does "minus the slope" mean when there are three directions you could walk in?  And what happens when there are *two* planets, and each one's potential energy depends on where the other one is?

That's next time.